In [5]:
import json
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools.tools import LLMQuery
import gradio as gr
from ai_tools.tools import MODEL_DICT
import os
import pandas as pd
import io

In [6]:
SYSTEM_PROMPT = """### ROLE
You are the **Mock Data Generator Assistant**, a specialized AI designed to generate high-fidelity, domain-specific mock datasets based on vague or specific business problem statements.

### OBJECTIVE
Your goal is to interpret a user's business scenario (e.g., "HR attrition analysis," "Supply chain logistics," "E-commerce transaction logs") and generate a realistic dataset in **strict JSON format**.

### OPERATIONAL RULES

1.  **Analyze the Domain:** deeply understand the industry implied by the prompt. If the user asks for "hospital patients," include medical-specific fields like `diagnosis_code`, `admission_date`, `insurance_provider`, and `blood_type`.
2.  **Infer Attributes:** Do not wait for the user to list columns. You must infer the most valuable 10-15 attributes that a data scientist or developer would need to solve the specific business problem.
3.  **Enforce Realism:**
    * Do not use placeholder values like "User1", "Test Data", or "asdf".
    * Use realistic names, addresses, UUIDs, timestamps, and domain-specific jargon (e.g., SKU numbers, ICD-10 codes, Stock tickers).
    * Ensure logical consistency (e.g., `delivery_date` must be after `order_date`; `age` must match `date_of_birth`).
4.  **Volume:** Unless specified otherwise, generate **10 records** to provide a representative sample.


### FORMATTING CONSTRAINTS (CRITICAL)
*   **Output Structure:** You must return a **flat JSON array** of objects (e.g., `[{"id": 1, ...}, {"id": 2, ...}]`).
*   **Root Element:** The root element MUST be the array `[...]`.
*   **NO WRAPPERS:** Do NOT wrap the array in an object. Do NOT use keys like "data", "records", "results", "analysis" or "schema".
*   **No Markdown:** Do NOT use markdown code blocks (like ```json). Just the raw JSON string.
*   **No Chatter:** Do not provide introductions, explanations, or "Here is the data". Start immediately with `[` and end with `]`.
*   **Consistency:** Ensure all objects in the array have the exact same set of keys.

### ERROR HANDLING
If the user request is gibberish or unrelated to data generation, try to interpret his intent and generate a dataset based on it.
### EXAMPLES
**User Input:** "I need to test a fraud detection system for credit cards."
**System Output:**
[
  {
    "transaction_id": "550e8400-e29b-41d4-a716-446655440000",
    "timestamp": "2023-10-27T14:30:00Z",
    "amount": 1250.00,
    "currency": "USD",
    "merchant": "Global Electronics",
    "is_flagged": true
  },
  {
    "transaction_id": "998e8400-e29b-41d4-a716-446655449999",
    "timestamp": "2023-10-27T14:35:00Z",
    "amount": 45.00,
    "currency": "USD",
    "merchant": "Local Cafe",
    "is_flagged": false
  }
]"""

**FRONTIER MODELS**

In [7]:
mock_data_generator_frontier = LLMQuery(
    system_prompt= SYSTEM_PROMPT,
    json_format=True
)

In [8]:
mock_data_generator_frontier.display_chat_history()

In [9]:

# Flatten model list for the dropdown
all_models = []
for models in MODEL_DICT.values():
    all_models.extend(list(models))
all_models.sort()

def generate_data(user_input, model_name):
    """
    Generates mock data based on user input and selected model.
    Returns a dataframe and a path to the CSV file.
    """
    if not user_input:
        return None, None

    try:
        # Query the LLM
        response = mock_data_generator_frontier.query(
            user_input, model=model_name, display_output=False, use_history=False
        )
        

        print(response)
        data = json.loads(response)
        # Ensure data is a list of objects for DataFrame
        if isinstance(data, dict):
            # If a single object is returned, wrap it in a list
            data = [data]

        df = pd.json_normalize(data)

        # Save to CSV for download
        # We save to a temporary file or a static one;
        # overwriting "generated_mock_data.csv" is fine for single user demo
        csv_filename = os.path.abspath(os.path.join(
            os.getcwd(), "mock_data.csv"
        ))
        df.to_csv(csv_filename, index=False)

        return df, csv_filename

    except Exception as e:
        # Return an error dataframe
        error_df = pd.DataFrame({"Error": [f"Failed to generate data: {str(e)}"]})
        return error_df, None


# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎲 Frontier Models - Mock Data Generator")
    gr.Markdown(
        "Describe the data you need, select a model, and generate a downloadable dataset."
    )

    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=all_models,
                value="gemini-flash-latest",
                label="Select Model",
                info="Choose the LLM to generate your data.",
                interactive=True,
            )
        with gr.Column(scale=3):
            pass  # Spacer

    user_input = gr.Textbox(
        label="Data Description",
        placeholder="e.g., I need a dataset of 10 fake transactions for a fraud detection system by extracting json...",
        lines=2,
    )

    with gr.Row():
        generate_btn = gr.Button("🚀 Generate Data", variant="primary")
        stop_btn = gr.Button("🛑 Stop", variant="stop")

    output_df = gr.DataFrame(label="Generated Data", interactive=False, wrap=True)

    with gr.Row():
        download_btn = gr.DownloadButton("📥 Download CSV")

    # Event wiring
    # Click button
    click_event = generate_btn.click(
        fn=generate_data,
        inputs=[user_input, model_selector],
        outputs=[output_df, download_btn],
    )

    # Stop button wiring
    stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[click_event])

    # Press Enter (submit)
    submit_event = user_input.submit(
        fn=generate_data,
        inputs=[user_input, model_selector],
        outputs=[output_df, download_btn],
    )

    # Allow stopping submit event as well
    stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[submit_event])

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[
  {
    "record_id": 1001,
    "year": 2023,
    "state_de": "Nordrhein-Westfalen",
    "city_de": "Köln",
    "total_population": 18131000,
    "density_per_sq_km": 529.0,
    "gender_ratio": 95.8,
    "age_group_0_14_percent": 13.5,
    "age_group_65_plus_percent": 22.1,
    "migration_balance_2022": 85000,
    "foreign_nationals_percent": 18.5,
    "unemployment_rate_percent": 6.8,
    "birth_rate_per_1000": 9.1
  },
  {
    "record_id": 1002,
    "year": 2022,
    "state_de": "Bayern",
    "city_de": "München",
    "total_population": 13140000,
    "density_per_sq_km": 186.0,
    "gender_ratio": 96.5,
    "age_group_0_14_percent": 14.2,
    "age_group_65_plus_percent": 20.5,
    "migration_balance_2022": 72000,
    "foreign_nationals_percent": 15.1,
    "unemployment_rate_percent": 3.1,
    "birth_rate_per_1000": 9.7
  },
  {
    "record_id": 1003,
    "year": 2023,
    "state_de": "Berlin",
    "city_de": "Berlin",
    "total_population": 3760000,
    "density_per_sq_km": 4200.0